In [ ]:
"""
Interactive LASY TE10 pulse inspector — uses the exact same Profile classes
as inputs_microwave_vacuum_te10_short_pulse.py.

Requires:  numpy  matplotlib  ipywidgets  lasy
Run with:  %matplotlib widget   (ipympl) or  %matplotlib notebook
"""

import numpy as np
import matplotlib.pyplot as plt
%matplotlib widget
import ipywidgets as widgets
from IPython.display import display

from lasy.laser import Laser
from lasy.profiles.profile import Profile

In [ ]:
import lasy

In [ ]:
print(lasy.__file__)

In [ ]:
print(lasy.__version__)

In [ ]:
# ── Physical constants ────────────────────────────────────────────────────────
c   = 299_792_458.0
mu0 = 1.25663706212e-6

# ── Profile classes (verbatim from the simulation script) ─────────────────────
class TE10CWProfile(Profile):
    """TE10 mode: Gaussian ramp-up → constant (CW) envelope."""
    def __init__(self, wavelength, pol, E0, a_wg, ramp_time, tau):
        super().__init__(wavelength, pol)
        self.E0 = E0;  self.a_wg = a_wg
        self.ramp_time = ramp_time;  self.tau = tau

    def evaluate(self, x, y, t):
        spatial  = self.E0 * np.cos(np.pi * x / self.a_wg)
        envelope = np.where(t < self.ramp_time,
                            1.0 - np.exp(-(t / self.tau)**2), 1.0)
        return (spatial * envelope).astype(complex)


class TE10PulseProfile(Profile):
    """TE10 mode: Gaussian pulse envelope."""
    def __init__(self, wavelength, pol, E0, a_wg, t_peak, tau):
        super().__init__(wavelength, pol)
        self.E0 = E0;  self.a_wg = a_wg
        self.t_peak = t_peak;  self.tau = tau

    def evaluate(self, x, y, t):
        spatial  = self.E0 * np.cos(np.pi * x / self.a_wg)
        envelope = np.exp(-((t - self.t_peak) / self.tau) ** 2)
        return (spatial * envelope).astype(complex)


# ── Default parameters ────────────────────────────────────────────────────────
DEFAULTS = dict(
    freq            = 67.0,    # GHz
    E0              = 1.0,     # MV/m
    a_horn          = 3.7592,  # mm
    pulse_t_peak    = 0.130,   # ns
    pulse_fwhm      = 0.050,   # ns
    cw_ramp_periods = 20,
)

# ── Widgets ───────────────────────────────────────────────────────────────────
style  = {'description_width': '160px'}
layout = widgets.Layout(width='480px')

mode_toggle = widgets.ToggleButtons(
    options=['pulse', 'cw'], value='pulse',
    description='Excitation mode:', style={'description_width': '130px'},
)

s_freq  = widgets.FloatSlider(value=DEFAULTS['freq'],   min=10,    max=120,  step=0.5,   description='Frequency (GHz)', style=style, layout=layout, continuous_update=True)
s_E0    = widgets.FloatSlider(value=DEFAULTS['E0'],     min=0.1,   max=10,   step=0.1,   description='E₀ (MV/m)',       style=style, layout=layout, continuous_update=True)
s_a     = widgets.FloatSlider(value=DEFAULTS['a_horn'], min=1.0,   max=20,   step=0.1,   description='a_horn (mm)',      style=style, layout=layout, continuous_update=True)
s_tpeak = widgets.FloatSlider(value=DEFAULTS['pulse_t_peak'], min=0.01, max=1.0,  step=0.01,  description='t_peak (ns)', style=style, layout=layout, continuous_update=True)
s_fwhm  = widgets.FloatSlider(value=DEFAULTS['pulse_fwhm'],   min=0.005, max=0.5, step=0.005, description='FWHM (ns)',   style=style, layout=layout, continuous_update=True)
s_ramp  = widgets.IntSlider(value=DEFAULTS['cw_ramp_periods'], min=1, max=100, step=1,    description='Ramp periods',    style=style, layout=layout, continuous_update=True)

pulse_box = widgets.VBox([s_tpeak, s_fwhm])
cw_box    = widgets.VBox([s_ramp])
param_box = widgets.VBox([pulse_box])
info_out  = widgets.Output()

# ── Figure ────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
fig.subplots_adjust(hspace=0.35)

line_env,  = axes[0].plot([], [], color='tab:orange', lw=2,   label='Envelope  |Re[E]|')
line_full, = axes[1].plot([], [], color='tab:blue',   lw=0.8, label='Re[E]·cos(ωt)')
line_ep,   = axes[1].plot([], [], color='tab:orange', lw=1.5, ls='--', alpha=0.8)
line_em,   = axes[1].plot([], [], color='tab:orange', lw=1.5, ls='--', alpha=0.8)

for ax in axes:
    ax.axhline(0, color='k', lw=0.5)
    ax.grid(True, alpha=0.35)

axes[0].set_ylabel("Ey envelope  (MV/m)", fontsize=10)
axes[0].legend(loc='upper right', fontsize=9)
axes[1].set_ylabel("Ey  (MV/m)",          fontsize=10)
axes[1].set_xlabel("Time  (ns)",           fontsize=10)
axes[1].legend(loc='upper right', fontsize=9)
title_txt = fig.suptitle("", fontsize=11)

plt.ion()
plt.show()

# ── Update ────────────────────────────────────────────────────────────────────
def update(_=None):
    freq_hz  = s_freq.value  * 1e9
    E0_vm    = s_E0.value    * 1e6
    a_m      = s_a.value     * 1e-3
    mode     = mode_toggle.value
    wl       = c / freq_hz
    omega    = 2 * np.pi * freq_hz
    k0       = omega / c
    fc       = c / (2 * a_m)
    beta     = np.sqrt(max(k0**2 - (np.pi / a_m)**2, 0.0))
    Z_TE10   = (omega * mu0 / beta) if beta > 0 else np.inf
    T_rf     = 1.0 / freq_hz

    # ── Build the same LASY objects as the simulation script ─────────────────
    if mode == 'pulse':
        tau    = s_fwhm.value  * 1e-9 / (2 * np.sqrt(np.log(2)))
        t_peak = s_tpeak.value * 1e-9

        profile = TE10PulseProfile(
            wavelength=wl, pol=(0, 1), E0=E0_vm, a_wg=a_m,
            t_peak=t_peak, tau=tau,
        )
        t_min = max(0.0, t_peak - 5 * tau)
        t_max = t_peak + 5 * tau
        nt    = max(64, int((t_max - t_min) * freq_hz * 10))
        mode_lbl = (f"Gaussian pulse  t_peak={t_peak*1e9:.3f} ns  "
                    f"FWHM={s_fwhm.value*1e3:.1f} ps  BW≈{1/(tau*1e9):.1f} GHz")

    else:
        ramp_p = s_ramp.value
        ramp_t = ramp_p * T_rf
        ramp_τ = ramp_t / 2.15

        profile = TE10CWProfile(
            wavelength=wl, pol=(0, 1), E0=E0_vm, a_wg=a_m,
            ramp_time=ramp_t, tau=ramp_τ,
        )
        t_min = 0.0
        t_max = ramp_t * 2
        # Nyquist: at least 10 points per RF period across the full window
        nt    = max(256, int(t_max * freq_hz * 10))
        mode_lbl = f"CW ramp  {ramp_p} periods = {ramp_t*1e12:.0f} ps"

    # Build Laser object exactly as in the simulation script (dim='xyt')
    laser = Laser(
        dim='xyt',
        lo=[-(a_m/2), -(a_m/4), t_min],
        hi=[ (a_m/2),  (a_m/4), t_max],
        npoints=(4, 4, nt),      # tiny transverse grid — we only need t axis
        profile=profile,
    )

    # Extract the time axis and Ey at the waveguide centre
    # grid.get_temporal_field() returns (2, nx, ny, nt) complex envelope
    # axis 0 = polarisation: 0→x, 1→y  (pol=(0,1) means y-polarised → index 1)
    t_ax  = laser.grid.axes[2]                        # 1-D time array (s)
    # field shape is (n_pol, nx, nt) — pol=(0,1) means component index 1 is Ey
    # but LASY packs both components, so shape[0]=2 normally; here it's 4
    # → just take the component with the largest amplitude
    field = laser.grid.get_temporal_field()
    pol_idx = np.argmax([np.abs(field[i]).max() for i in range(field.shape[0])])
    env_centre = np.abs(field[pol_idx, field.shape[1]//2, :])

    # Reconstruct physical field: E(t) = Re[ envelope(t) · exp(iωt) ]
    carrier = env_centre * np.cos(omega * t_ax)
    t_ns    = t_ax * 1e9

    # ── Update plot ───────────────────────────────────────────────────────────
    ylim = max(env_centre.max(), 1e-9) / 1e6 * 1.15
    axes[0].set_xlim(t_ns[0], t_ns[-1])
    axes[1].set_xlim(t_ns[0], t_ns[-1])
    axes[0].set_ylim(-ylim * 0.05, ylim)
    axes[1].set_ylim(-ylim, ylim)

    line_env.set_data(t_ns, env_centre / 1e6)
    line_full.set_data(t_ns, carrier    / 1e6)
    line_ep.set_data(t_ns,  env_centre / 1e6)
    line_em.set_data(t_ns, -env_centre / 1e6)

    title_txt.set_text(
        f"TE10 Ey @ x=0  |  f={s_freq.value:.1f} GHz  |  {mode_lbl}\n"
        f"fc={fc*1e-9:.2f} GHz   Z_TE10={Z_TE10:.1f} Ω   E₀={s_E0.value:.1f} MV/m"
    )

    info_out.clear_output(wait=True)
    with info_out:
        flag = "✅ propagating" if freq_hz > fc else "⚠️  EVANESCENT (f < fc)"
        print(f"  Cutoff  : {fc*1e-9:.3f} GHz  →  {flag}")
        if beta > 0:
            vg = c * beta / k0
            lg = 2 * np.pi / beta
            print(f"  λ_guide : {lg*1e3:.2f} mm   vg = {vg/c:.4f} c   Z_TE10 = {Z_TE10:.1f} Ω")
        if mode == 'pulse':
            print(f"  τ       : {tau*1e12:.2f} ps   BW ≈ {1/(tau*1e9):.1f} GHz")
        print(f"  LASY field shape: {tuple(field.shape)}  pol_idx={pol_idx}  "
              f"t=[{t_ax[0]*1e9:.3f}, {t_ax[-1]*1e9:.3f}] ns")

    fig.canvas.draw_idle()

# ── Callbacks ─────────────────────────────────────────────────────────────────
def on_mode_change(change):
    param_box.children = (pulse_box,) if change['new'] == 'pulse' else (cw_box,)
    update()

mode_toggle.observe(on_mode_change, names='value')
for w in [s_freq, s_E0, s_a, s_tpeak, s_fwhm, s_ramp]:
    w.observe(update, names='value')

# ── Display ───────────────────────────────────────────────────────────────────
ui = widgets.VBox([
    mode_toggle,
    widgets.HBox([
        widgets.VBox([s_freq, s_E0, s_a]),
        param_box,
    ]),
    info_out,
])

display(ui)
update()